# 00 · Windows daily Left-check setup

Use this notebook once to prepare a no-admin Windows automation for `01_left_snapshot_counts.ipynb`.

Runtime chain:

```text
Windows Task Scheduler (daily 10:00)
  -> Git for Windows bash.exe
  -> scripts/run_left_snapshot.sh
  -> .venv/Scripts/python.exe
  -> scripts/run_notebook.py
  -> nbs/01_left_snapshot_counts.ipynb
  -> local immutable JSON event
  -> AWS CLI sync to the S3 prefix behind an External Volume
```

The recommended execution mode is **Python**, not `jupyter nbconvert --execute`.

Why:

- Notebook 01 contains ordinary Python cells and no notebook magics.
- The runner uses only the Python standard library to read and execute code cells.
- There is no Jupyter kernel startup, browser session, or notebook output rewrite.
- Notebook 01 remains the single business source; no separately maintained copy of its code is required.

Notebook 03 currently schedules the Databricks Right check at 09:00. If the Windows Left check starts at 10:00, change Notebook 03 to 10:30 or later before deploying that job.


## 1. Windows prerequisites

No administrator rights are required if the following software is already available to your user account:

1. Git for Windows, including `C:\Program Files\Git\bin\bash.exe`.
2. Python 3.11 or newer.
3. Network access to the Left database and the target S3 bucket.
4. AWS CLI configured with write access to the records prefix.
5. Permission to create a Task Scheduler task for your own Windows account.
6. A project path without spaces is strongly recommended, for example:

   ```text
   C:\Users\<username>\paper-orbit
   ```

Create the virtual environment from Git Bash:

```bash
cd /c/Users/<username>/paper-orbit
python -m venv .venv
source .venv/Scripts/activate
python -m pip install --upgrade pip
python -m pip install oracledb
```

Install `duckdb` as well only if a Left configuration uses DuckDB.

Set each side's `partition_type` to match the physical column: `date` / `timestamp` / `datetime` for native temporal columns, `string_dash` for `2025-04-01`, `string_compact` for string `20250401`, or `number` / `num` / `integer` / `int` / `numeric` for numeric `20250401`. The notebooks query the raw latest value and normalize it to an ISO date only for comparison.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "nbs":
    PROJECT_ROOT = PROJECT_ROOT.parent

LEFT_NOTEBOOK = PROJECT_ROOT / "nbs" / "01_left_snapshot_counts.ipynb"
RIGHT_JOB_NOTEBOOK = PROJECT_ROOT / "nbs" / "03_dbx_job_setup.ipynb"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "run_notebook.py"
SHELL_PATH = SCRIPTS_DIR / "run_left_snapshot.sh"
ENV_EXAMPLE_PATH = SCRIPTS_DIR / "count_watch.env.example"

# Git Bash path used inside the generated shell script.
# Edit this value before running the generation cell.
PROJECT_ROOT_BASH = "/c/Users/<username>/paper-orbit"

# Keep the real credential file outside the repository.
ENV_FILE_BASH = "/c/Users/<username>/.dtrack/count_watch.env"

assert LEFT_NOTEBOOK.exists(), LEFT_NOTEBOOK
assert RIGHT_JOB_NOTEBOOK.exists(), RIGHT_JOB_NOTEBOOK
print("project:", PROJECT_ROOT)
print("left notebook:", LEFT_NOTEBOOK)


In [ ]:
RUNNER_SOURCE = r'''#!/usr/bin/env python3
"""Execute the Python code cells of one .ipynb in a shared namespace."""

from __future__ import annotations

import argparse
import json
import os
import sys
import traceback
from pathlib import Path


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("notebook", type=Path)
    parser.add_argument("--project-root", type=Path, required=True)
    args = parser.parse_args()

    project_root = args.project_root.resolve()
    notebook = args.notebook.resolve()
    os.chdir(project_root)

    document = json.loads(notebook.read_text(encoding="utf-8"))
    namespace = {
        "__name__": "__main__",
        "__file__": str(notebook),
    }

    code_number = 0
    for cell_number, cell in enumerate(document.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue
        code_number += 1
        source = "".join(cell.get("source", []))
        if not source.strip():
            continue
        label = f"{notebook}:cell-{cell_number}"
        print(f"[runner] code cell {code_number}: {label}", flush=True)
        try:
            exec(compile(source, label, "exec"), namespace, namespace)
        except BaseException:
            print(f"[runner] failed: {label}", file=sys.stderr, flush=True)
            traceback.print_exc()
            return 1

    print(f"[runner] completed {code_number} code cell(s)", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''


In [ ]:
SHELL_SOURCE = f'''#!/usr/bin/env bash
set -Eeuo pipefail

project_root="{PROJECT_ROOT_BASH}"
env_file="{ENV_FILE_BASH}"
python_exe="$project_root/.venv/Scripts/python.exe"
runner="$project_root/scripts/run_notebook.py"
notebook="$project_root/nbs/01_left_snapshot_counts.ipynb"
records_root="$project_root/records/count_watch"
log_dir="$project_root/logs"
lock_dir="$project_root/.run_state/left_snapshot.lock"

mkdir -p "$log_dir" "$(dirname "$lock_dir")"

if ! mkdir "$lock_dir" 2>/dev/null; then
  printf '%s another Left check is already running\n' "$(date -Is)" \
    >> "$log_dir/left_snapshot_scheduler.log"
  exit 75
fi
trap 'rmdir "$lock_dir" 2>/dev/null || true' EXIT

if [[ ! -f "$env_file" ]]; then
  printf 'Missing environment file: %s\n' "$env_file" >&2
  exit 2
fi
if [[ ! -x "$python_exe" ]]; then
  printf 'Missing virtual-environment Python: %s\n' "$python_exe" >&2
  exit 2
fi

set -a
# shellcheck disable=SC1090
source "$env_file"
set +a

log_file="$log_dir/left_snapshot_$(date +%Y-%m-%d).log"
{{
  printf '\n===== %s =====\n' "$(date -Is)"
  "$python_exe" "$runner" "$notebook" --project-root "$project_root"
  aws s3 sync "$records_root/" "$DTRACK_RECORDS_URI/" --exclude "*.tmp" --only-show-errors
}} >> "$log_file" 2>&1

find "$log_dir" -type f -name 'left_snapshot_*.log' -mtime +30 -delete
'''


In [ ]:
ENV_EXAMPLE = r'''# Copy this file to:
#   C:\Users\<username>\.dtrack\count_watch.env
# Keep the real file outside Git.

DTRACK_COUNT_WATCH_CONFIG=nbs/table_list.example.json
DTRACK_RECORDS_ROOT=records/count_watch
DTRACK_RECORDS_URI=s3://<bucket>/<prefix>/count-watch

# Add one group for every Oracle conn_name used by left_conn_detail.
# Example for conn_name "pcds":
pcds_USR=<oracle-user>
pcds_PWD=<oracle-password>
pcds_DSN=<oracle-dsn>
'''


In [ ]:
if "<username>" in PROJECT_ROOT_BASH or "<username>" in ENV_FILE_BASH:
    raise ValueError("Set PROJECT_ROOT_BASH and ENV_FILE_BASH before generating scripts")

SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
RUNNER_PATH.write_text(RUNNER_SOURCE, encoding="utf-8", newline="\n")
SHELL_PATH.write_text(SHELL_SOURCE, encoding="utf-8", newline="\n")
ENV_EXAMPLE_PATH.write_text(ENV_EXAMPLE, encoding="utf-8", newline="\n")

RUNNER_PATH.chmod(0o755)
SHELL_PATH.chmod(0o755)

print("created:", RUNNER_PATH)
print("created:", SHELL_PATH)
print("created:", ENV_EXAMPLE_PATH)


## 2. Create the real environment file

After running the generation cell:

1. Create this user-only directory:

   ```text
   C:\Users\<username>\.dtrack
   ```

2. Copy `scripts/count_watch.env.example` to:

   ```text
   C:\Users\<username>\.dtrack\count_watch.env
   ```

3. Replace every placeholder.
4. Add one `<conn_name>_USR/PWD/DSN` group for every Oracle connection name stored in the Delta table.
5. Do not place the real file inside the repository.
6. Use Windows file properties or `icacls` to restrict the file to your account if company policy allows it.

The shell wrapper sources this file immediately before starting Python. Notebook 01 receives the values through `os.environ`.


## 3. Test manually before scheduling

Open Git Bash and run:

```bash
cd /c/Users/<username>/paper-orbit
./scripts/run_left_snapshot.sh
echo "exit=$?"
tail -n 100 logs/left_snapshot_$(date +%Y-%m-%d).log
```

Success criteria:

1. Exit code is `0`.
2. The log shows every Notebook 01 code cell completing.
3. A JSON event exists under `records/count_watch/query_date=.../pair_name=...`.
4. The same file is visible through the configured S3 prefix and External Volume.
5. Running again after a successful Left check does not add another event for that deterministic daily `check_id`. Failed attempts remain immutable and can be retried.

Do not create the scheduled task until this manual run succeeds.


## 4. Create the 10:00 Task Scheduler task

Use the GUI; it works without administrator rights when company policy permits user-owned tasks.

1. Press `Win+R`.
2. Run `taskschd.msc`.
3. Choose **Create Task** rather than Create Basic Task.
4. General tab:
   - Name: `DTrack Left Snapshot`
   - Select **Run only when user is logged on**.
   - Do not select **Run with highest privileges**.
5. Triggers tab:
   - New trigger
   - Daily
   - Start: `10:00:00 AM`
   - Enabled
6. Actions tab:
   - Action: **Start a program**
   - Program/script:

     ```text
     C:\Program Files\Git\bin\bash.exe
     ```

   - Add arguments:

     ```text
     --login -c "/c/Users/<username>/paper-orbit/scripts/run_left_snapshot.sh"
     ```

   - Start in:

     ```text
     C:\Users\<username>\paper-orbit
     ```

7. Conditions tab:
   - If this is a laptop, decide whether the task may run on battery.
   - Enable **Wake the computer to run this task** only if company policy permits it.
8. Settings tab:
   - Allow task to be run on demand.
   - If the task fails, restart every 10 minutes.
   - Attempt to restart 3 times.
   - If the task is already running, choose **Do not start a new instance**.
   - Stop the task if it runs longer than 2 hours.
9. Save the task.
10. Right-click it and choose **Run**.
11. Confirm **Last Run Result** is `0x0` and inspect the daily log.

If the project path contains spaces, move it to a path without spaces rather than adding another quoting layer.


## 5. Coordinate with Notebook 03

The Windows Left task begins at 10:00. The Databricks Right job must run later.

Recommended Notebook 03 setting:

```python
CRON = "0 30 10 * * ?"   # 10:30 America/New_York
```

After changing it, rerun `03_dbx_job_setup.ipynb` to update the Databricks job.

Daily order:

```text
10:00 Windows Task Scheduler -> Notebook 01 writes JSON, wrapper syncs S3
10:30 Databricks job         -> Notebook 02 ingests Volume and fills Right result
after 10:30 Dashboard        -> refreshes from the same Delta table
```

If the Left query can take longer than 30 minutes, move the Right job later. Notebook 02 processes all eligible pending rows, so a late Left row is not lost; it is processed by the next Right-job run.


## 6. Optional Jupyter execution

Jupyter remains possible, but is not recommended for the scheduled path:

```bash
jupyter nbconvert \
  --execute \
  --to notebook \
  --output-dir logs \
  nbs/01_left_snapshot_counts.ipynb
```

This requires Jupyter and a configured kernel and creates an executed notebook artifact. Use it for interactive diagnostics, not the default daily task.

For normal automation, keep:

```text
Task Scheduler -> .sh -> Python runner -> Notebook 01
```


## 7. Troubleshooting checklist

- `0x0`: task completed successfully.
- `0x1`: inspect the daily log for a Python or database error.
- `0x2`: environment file or virtual-environment Python is missing.
- Exit `75`: a previous run still owns the lock directory.
- No log file: verify the Task Scheduler action and Git Bash path.
- Works manually but not in Task Scheduler: use absolute paths and confirm **Start in**.
- Oracle import error: install `oracledb` into this project's `.venv`.
- AWS sync error: run `aws sts get-caller-identity` and confirm write access to `DTRACK_RECORDS_URI`.
- Volume sees no files: confirm the External Volume points at the same cloud prefix.
- Right result arrives a day late: schedule Notebook 03 later than the longest expected Left runtime.
- Stale lock after a forced shutdown: confirm no runner is active, then remove `.run_state/left_snapshot.lock`.
